# Methane dimer at 36 qubits - SQD

Reproducing the 36-qubit experiment in [arXiv:2410.09209](https://arxiv.org/abs/2410.09209),
*Accurate quantum-centric simulations of supramolecular interactions* (Commun. Phys. 2025).

**CAS(16e,16o)/aug-cc-pVQZ | 165,636,900 determinants | 32 + 4 ancilla = 36 qubits**

Run this on Colab because `pyscf` and `ffsim` ship no Windows wheels.

> **Set Runtime -> Change runtime type -> High-RAM before the CASCI cell.**
> The exact reference needs ~15 GiB; free Colab will be OOM-killed.

## 1 - Install (pinned)

In [ ]:
!pip install -q pyscf==2.14.0 ffsim==0.0.84 'qiskit>=2.0,<3' qiskit-addon-sqd==0.13.1
import pyscf, ffsim, qiskit, qiskit_addon_sqd
print('pyscf', pyscf.__version__, '| ffsim', ffsim.__version__,
      '| qiskit', qiskit.__version__, '| sqd', qiskit_addon_sqd.__version__)

## 2 - Get the code
Clone your repo, or upload the `methane_dimer_36/` folder into the session.

In [ ]:
import os, sys
# If you have pushed the repo:
# !git clone https://github.com/<you>/<repo>.git
FOLDER = 'methane_dimer_36'
if os.path.isdir(FOLDER):
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())
print('cwd =', os.getcwd())

## 3 - Self-test and cost model
Neither needs the chemistry stack. Run them first: they catch version drift
before any CPU time is spent.

In [ ]:
!python run.py selftest --quiet
!python run.py plan

## 3b - END-TO-END VALIDATION (run this before anything expensive)

Runs the **entire chemistry pipeline** - RHF, AVAS, cache round-trip, CCSD, LUCJ,
ffsim sampling, SQD, variance, ablation, binding energy - on a tiny STO-3G active
space. Same molecule, same functions, seconds to run.

The load-bearing check: when the subspace saturates the CAS, **SQD must equal
CASCI to 1e-8 Ha** - solver precision, not chemical accuracy. If the SQD and
CASCI Hamiltonians ever differ, this catches it immediately.

**If this fails, stop and fix it. Nothing below will be right.**

In [ ]:
!python run.py validate

## 4 - Geometry
The paper publishes no coordinates and never names the monomer orientation.
D3d is the default; change it here and every downstream cache key changes with it.

In [ ]:
ORIENTATION = 'd3d'
DISTANCE    = 3.638   # the paper's extra point, and its equilibrium

import geometry, paper
atoms = geometry.methane_dimer(DISTANCE, orientation_name=ORIENTATION)
geometry.check_geometry(atoms, expected_distance=DISTANCE)
print(geometry.to_xyz(atoms, f'methane dimer {ORIENTATION} R={DISTANCE}'))

## 5 - Hamiltonian: RHF/aug-cc-pVQZ + AVAS
528 basis functions, so density fitting is on. AVAS on `C[2s,2p], H[1s]` must
land on exactly (16e,16o); `chemistry.py` asserts it rather than tuning to it.

In [ ]:
from pathlib import Path
import chemistry, spaces

CACHE = Path('data/cache')
spec  = chemistry.SystemSpec(distance=DISTANCE, orientation=ORIENTATION)
mol_data = chemistry.load_or_build(spec, CACHE, verbose=4)

print(f'HF           {mol_data.hf_energy:.10f} Ha')
print(f'active       ({sum(mol_data.nelec)}e,{mol_data.norb}o)')
print(f'determinants {spaces.n_determinants(mol_data.norb, mol_data.nelec):,}')

## 6 - CCSD amplitudes and the LUCJ ansatz
Cheap: `MolecularData.scf` round-trips the *active-space* integrals through an
FCIDUMP, so this is a 16-orbital CCSD, not a 528-orbital one.

In [ ]:
import ansatz, reference

e_ccsd = reference.run_ccsd(mol_data, store_amplitudes=True)
print(f'CCSD (active space) {e_ccsd:.10f} Ha')

layout = ansatz.heavy_hex_layout(mol_data.norb, n_reps=2)
ansatz.validate_layout(layout)      # asserts 32 + 4 = 36
print(layout.to_dict())

operator = ansatz.build_operator(mol_data, layout)
circuit  = ansatz.build_circuit(mol_data, operator)
print(f'circuit: {circuit.num_qubits} qubits, depth {circuit.depth()}')

## 7 - Sample
`FfsimSampler` implements the SamplerV2 interface but simulates inside the (8,8)
sector: 2.47 GiB rather than the 64 GiB a dense 32-qubit statevector would need.

Swap in `sampling.HardwareSampler(backend)` for a real device. Nothing downstream changes.

In [ ]:
import sampling

SHOTS = paper.TOTAL_SAMPLES      # 200,000
sampler = sampling.NoiselessSampler(mol_data.norb, mol_data.nelec, seed=12345)
sampled = sampler.sample(circuit, SHOTS)

valid    = sampling.valid_configuration_fraction(sampled.bit_array, mol_data.norb, mol_data.nelec)
baseline = sampling.random_validity_probability(mol_data.norb, mol_data.nelec)
print(f'valid {valid:.2%}   random baseline {baseline:.2%}')
print()
print('NOTE: at half filling the random baseline is percent-level, so validity')
print('      fraction is a weak diagnostic here. Cell 10 is the real control.')

## 8 - SQD
Start on the `extrapolation-low` rung (|chi_b| = 9e3). The `converged` rung
reproduces Table II verbatim but needs a large-memory machine - see `run.py plan`.

In [ ]:
import sqd

rung   = spaces.rung('extrapolation-low')
config = sqd.SqdConfig(samples_per_batch=rung.samples_per_batch,
                       n_batches=rung.n_batches,
                       max_iterations=paper.RECOVERY_STEPS,
                       max_dim=rung.max_dim, seed=12345)

result = sqd.run(mol_data, sampled.bit_array, config, source='noiseless')
print(f'\nSQD  E = {result.energy:.10f} Ha')
print(f'     d = {result.subspace_dimension:,} ({result.subspace_fraction:.2%} of CAS)')

## 9 - CASCI reference (needs High-RAM)
165,636,900 determinants. One CI vector is 961 MiB; budget ~15 GiB.

In [ ]:
import binding

e_casci = reference.run_casci(mol_data, verbose=4)
print(f'CASCI {e_casci:.10f} Ha')

binding.variational_check(result.energy, e_casci)   # SQD may not fall below CASCI
agreement = binding.Agreement(result.energy, e_casci)
print(agreement)
print(f'paper target at |chi_b|=20e3: {paper.SQD_VS_CASCI_TARGET_KCAL} kcal/mol')

## 10 - Ablation: the measurement that matters
Uniform random configurations at **matched subspace dimension**. The energy gap
is the quantum layer's contribution, as a number. A control at a different
dimension compares two things at once and settles nothing.

In [ ]:
control_cfg = sqd.ablation_config(result, config)
uniform     = sampling.UniformSampler(mol_data.norb, mol_data.nelec, seed=12345)
control     = sqd.run(mol_data, uniform.sample(shots=SHOTS).bit_array,
                      control_cfg, source='uniform')

gap = (control.energy - result.energy) / binding.MILLIHARTREE
print(f'SQD      {result.energy:.10f} Ha   d = {result.subspace_dimension:,}')
print(f'uniform  {control.energy:.10f} Ha   d = {control.subspace_dimension:,}')
print(f'gap      {gap:+.4f} mHa')

## 11 - Scan the PES, verify, report
Binding energy is `E(R) - E(48 A)` (paper Eq. 2), so the 48 A point is required,
not optional. `--all` includes it.

In [ ]:
!python run.py sqd       --all --rung extrapolation-low
!python run.py reference --all
!python run.py verify
!python run.py report